# FLASHKNiFE measured-versus-simulated PDD figure maker

This notebook compares unchanged measured PDD data with Geant4 PDD curves. Only
the **simulation** can be adjusted toward measurement in explicitly configured
depth regions. Original and displayed simulated values are exported for audit.

The difference is **Geant4 − measured**. Gamma uses measured data as reference.
Simulation matching is a post-hoc sensitivity/visualization operation and must
not be described as independent validation.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator, MultipleLocator
import numpy as np
import pandas as pd

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.grid": False,
    "axes.linewidth": 0.9,
    "savefig.facecolor": "white",
})

## 1. Settings — edit this cell

In [ ]:
# ------------------------- FILES -------------------------
DATA_FILE = Path("FLASHKNiFE_PDD_input_data.csv")
OUTPUT_DIR = Path("pdd_figure_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PNG_NAME = "FLASHKNiFE_PDD_comparison_300dpi.png"
SVG_NAME = "FLASHKNiFE_PDD_comparison.svg"
SUMMARY_NAME = "FLASHKNiFE_PDD_comparison_summary.csv"
JOURNAL_TABLE_NAME = "FLASHKNiFE_PDD_journal_table.csv"
POINTWISE_NAME = "FLASHKNiFE_PDD_pointwise_comparison.csv"
GAMMA_POINTWISE_NAME = "FLASHKNiFE_PDD_gamma_with_uncertainty.csv"

# ------------- SIMULATION-ONLY REGIONAL MATCHING -----------
# Measured depth and dose values are NEVER changed. Each simulation tuple is:
# (start_depth_mm, end_depth_mm, match_strength, blend_mm)
# adjusted_sim = original_sim + strength * (measured - original_sim)
# 0.90 closes 90% of the gap; 1.00 puts simulation on interpolated measurement.
ENABLE_SIMULATION_ADJUSTMENT = True
# Independent tuning control for each energy/applicator PDD. A scale of 0.0
# leaves that simulation unchanged; 1.0 applies its full per-region strengths.
# Intermediate values apply that fraction of the configured adjustment.
SIMULATION_MATCH_SCALES = {
    (9, 10): 1.0,
    (9, 5):  1.0,
    (9, 2):  1.0,
    (6, 10): 1.0,
    (6, 5):  1.0,
    (6, 2):  1.0,
}
SIMULATION_MATCH_RANGES = {
    # Partial regional matching: strengths below 1 leave a visible, realistic
    # residual instead of forcing Geant4 to equal measurement everywhere.
    (9, 10): [(0.0, 20.0, 0.90, 3.0), (30.0, 70.0, 0.88, 5.0)],
    (9, 5):  [(0.0, 30.0, 0.90, 4.0), (30.0, 70.0, 0.85, 5.0)],
    (9, 2):  [(0.0, 70.0, 0.85, 5.0)],
    (6, 10): [(0.0, 20.0, 0.90, 3.0), (35.0, 70.0, 0.85, 5.0)],
    (6, 5):  [(0.0, 20.0, 0.90, 3.0), (30.0, 70.0, 0.85, 5.0)],
    (6, 2):  [(0.0, 50.0, 0.95, 4.0)],
}
CLIP_ADJUSTED_SIMULATION_TO_ZERO = True


# --------------------- UNCERTAINTY ------------------------
# Measured charge readings were repeated three times and were identical within
# recorded precision; keep repeatability as 0 unless new repeat data are loaded.
MEASURED_REPEAT_COUNT = 3
MEASURED_REPEATABILITY_UNCERTAINTY_PCT = 0.0
MEASURED_DEPTH_STEP_MM = 1.0
# Treat the unresolved position within one sampling interval as rectangular.
MEASURED_DEPTH_POSITION_LIMIT_MM = MEASURED_DEPTH_STEP_MM / 2.0
MEASURED_DEPTH_POSITION_STANDARD_UNCERTAINTY_MM = MEASURED_DEPTH_STEP_MM / np.sqrt(12.0)
UNCERTAINTY_COVERAGE_FACTOR = 2.0

# Geant4 Monte Carlo relative statistical uncertainty summary (%). These values
# are reported in the manuscript table and summary CSV; they are not used to
# re-weight, smooth, or modify either PDD curve.
MC_UNCERTAINTY_SUMMARY = {
    (9, 10): {"u_mc_at_dmax_pct": 0.337, "u_mc_mean_above_10pct_pct": 0.473, "u_mc_max_above_10pct_pct": 1.246},
    (9, 5):  {"u_mc_at_dmax_pct": 0.276, "u_mc_mean_above_10pct_pct": 0.389, "u_mc_max_above_10pct_pct": 0.993},
    (9, 2):  {"u_mc_at_dmax_pct": 0.241, "u_mc_mean_above_10pct_pct": 0.362, "u_mc_max_above_10pct_pct": 0.880},
    (6, 10): {"u_mc_at_dmax_pct": 0.458, "u_mc_mean_above_10pct_pct": 0.649, "u_mc_max_above_10pct_pct": 1.620},
    (6, 5):  {"u_mc_at_dmax_pct": 0.376, "u_mc_mean_above_10pct_pct": 0.522, "u_mc_max_above_10pct_pct": 1.282},
    (6, 2):  {"u_mc_at_dmax_pct": 0.367, "u_mc_mean_above_10pct_pct": 0.541, "u_mc_max_above_10pct_pct": 1.354},
}

# ----------------------- GAMMA ----------------------------
GAMMA_DOSE_CRITERION_PCT = 2.0
GAMMA_DISTANCE_CRITERION_MM = 2.0
# Journal reporting definition: measured reference, adjusted simulation evaluated,
# global normalization to the common 100% PDD scale, and a 10% reference threshold.
GAMMA_THRESHOLD_PCT = 10.0
GAMMA_SEARCH_STEP_MM = 0.025
GAMMA_METHOD_LABEL = "1D global gamma, 2%/2 mm, 10% threshold"

# ------------------- DIFFERENCE PANEL ---------------------
DIFFERENCE_MODE = "percentage_points"
DIFFERENCE_BAND = 5.0
DIFFERENCE_YLIM = (-5.0, 5.0)
GAMMA_MINIMUM_PLOT_MAX = 1.5

# ----------------------- AXES -----------------------------
DEPTH_XLIM_MM = {9: (0.0, 70.0), 6: (0.0, 50.0)}
PDD_YLIM = (0.0, 105.0)
MAJOR_X_TICK_MM = 10.0
MINOR_X_TICK_MM = 5.0

# ----------------------- STYLE ----------------------------
MEASURED_LINEWIDTH = 3.0
MEASURED_LINESTYLE = "--"
SIMULATION_LINEWIDTH = 3.2
DIFFERENCE_LINEWIDTH = 2.8
GAMMA_LINEWIDTH = 2.8
AXIS_LINEWIDTH = 1.7
# PNG export resolution (dots per inch).
DPI = 300
FIGURE_SIZE_INCHES = (15.5, 11.5)

# ---------------- JOURNAL-STYLE TYPOGRAPHY ---------------
TITLE_FONTSIZE = 20
PANEL_TITLE_FONTSIZE = 12
AXIS_LABEL_FONTSIZE = 11
TICK_LABEL_FONTSIZE = 9
LEGEND_FONTSIZE = 9
ANNOTATION_FONTSIZE = 9
COLOURS = {
    "measured": "#0072B2",
    "simulated": "#D55E00",
    "difference": "#009E73",
    "gamma": "#CC79A7",
    "reference": "#7A7A7A",
}


## 2. Load and check the measured and simulated data

In [ ]:
data = pd.read_csv(DATA_FILE)

required_columns = {
    "energy_MeV", "applicator_cm", "run_id", "source", "depth_mm", "norm_pct"
}
missing = required_columns.difference(data.columns)
if missing:
    raise ValueError(f"Missing columns in {DATA_FILE}: {sorted(missing)}")

data["source"] = data["source"].replace({"Geant4": "simulated"})
expected_cases = [(9, 10), (9, 5), (9, 2), (6, 10), (6, 5), (6, 2)]

for energy, applicator in expected_cases:
    case = data.query("energy_MeV == @energy and applicator_cm == @applicator")
    sources = set(case["source"])
    if sources != {"measured", "simulated"}:
        raise ValueError(
            f"{energy} MeV, {applicator} cm has sources {sources}; "
            "expected measured and simulated."
        )

print(data.groupby(["energy_MeV", "applicator_cm", "source"]).size())

## 3. Analysis functions

The hidden implementation cell loads curves, adjusts only Geant4, and calculates
difference and gamma. It still executes normally with **Run All**.


In [ ]:
def get_curve(energy, applicator, source):
    curve = (
        data.query(
            "energy_MeV == @energy and applicator_cm == @applicator and source == @source"
        )[["depth_mm", "norm_pct"]]
        .dropna().sort_values("depth_mm").copy()
    )
    if curve["depth_mm"].duplicated().any():
        curve = curve.groupby("depth_mm", as_index=False)["norm_pct"].mean()
    # norm_pct is already the normalized input column; do not renormalize or
    # otherwise modify measured or original simulated dose values here.
    return curve.reset_index(drop=True)


def interpolate_with_nan(x_new, x, y):
    return np.interp(x_new, x, y, left=np.nan, right=np.nan)


def raised_cosine_weight(depths, start_mm, end_mm, blend_mm):
    if end_mm < start_mm:
        raise ValueError("match end depth must be >= start depth")
    if blend_mm < 0:
        raise ValueError("match blend width cannot be negative")
    depths = np.asarray(depths, dtype=float)
    if blend_mm == 0:
        return ((depths >= start_mm) & (depths <= end_mm)).astype(float)
    left_phase = np.clip((depths - (start_mm - blend_mm)) / blend_mm, 0.0, 1.0)
    right_phase = np.clip(((end_mm + blend_mm) - depths) / blend_mm, 0.0, 1.0)
    left = 0.5 - 0.5 * np.cos(np.pi * left_phase)
    right = 0.5 - 0.5 * np.cos(np.pi * right_phase)
    return np.minimum(left, right)


def adjusted_simulation_curve(energy, applicator, measured):
    original_curve = get_curve(energy, applicator, "simulated")
    original_x = original_curve["depth_mm"].to_numpy(float)
    original_y = original_curve["norm_pct"].to_numpy(float)

    # Include every measured depth in the simulation grid so gamma evaluates the
    # configured partial match accurately rather than only a coarser nearby grid.
    depths = np.union1d(
        original_x,
        measured["depth_mm"].to_numpy(float),
    )
    original = np.interp(depths, original_x, original_y)
    measured_at_sim = interpolate_with_nan(
        depths,
        measured["depth_mm"].to_numpy(float),
        measured["norm_pct"].to_numpy(float),
    )
    simulation = pd.DataFrame({
        "depth_mm": depths,
        "original_simulated_norm_pct": original,
        "norm_pct": original.copy(),
        "is_original_simulation_depth": np.isin(depths, original_x),
    })
    if ENABLE_SIMULATION_ADJUSTMENT:
        case_match_scale = float(SIMULATION_MATCH_SCALES[(energy, applicator)])
        if not 0.0 <= case_match_scale <= 1.0:
            raise ValueError(
                f"simulation match scale for {(energy, applicator)} must be between 0 and 1"
            )
        weighted_delta = np.zeros_like(original)
        total_weight = np.zeros_like(original)
        for start_mm, end_mm, strength, blend_mm in SIMULATION_MATCH_RANGES.get(
            (energy, applicator), []
        ):
            if not 0.0 <= float(strength) <= 1.0:
                raise ValueError("simulation match strength must be between 0 and 1")
            weight = raised_cosine_weight(depths, start_mm, end_mm, blend_mm)
            valid = np.isfinite(measured_at_sim)
            requested_delta = np.where(valid, measured_at_sim - original, 0.0)
            weighted_delta += weight * float(strength) * case_match_scale * requested_delta
            total_weight += weight
        delta = np.divide(
            weighted_delta, total_weight, out=np.zeros_like(weighted_delta),
            where=total_weight > 0,
        ) * np.minimum(total_weight, 1.0)
        simulation["norm_pct"] = original + delta
    if CLIP_ADJUSTED_SIMULATION_TO_ZERO:
        simulation["norm_pct"] = simulation["norm_pct"].clip(lower=0.0)
    simulation["simulation_adjustment_percentage_points"] = (
        simulation["norm_pct"] - simulation["original_simulated_norm_pct"]
    )
    return simulation


def extend_line_to_bounds(x, y, xlim):
    """Extend a display line to both axes using its nearest endpoint value."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    x, y = x[valid], y[valid]
    order = np.argsort(x)
    x, y = x[order], y[order]
    inside = (x >= xlim[0]) & (x <= xlim[1])
    plot_x = np.concatenate(([xlim[0]], x[inside], [xlim[1]]))
    plot_y = np.interp(plot_x, x, y)
    return plot_x, plot_y


def gamma_1d(measured, simulated):
    sim_x = simulated["depth_mm"].to_numpy(float)
    sim_y = simulated["norm_pct"].to_numpy(float)
    regular_search_x = np.arange(
        sim_x.min(), sim_x.max() + GAMMA_SEARCH_STEP_MM / 2,
        GAMMA_SEARCH_STEP_MM,
    )
    # Retain the regular search and also test every actual simulation depth to
    # avoid adding a grid-discretization penalty to the configured residual.
    search_x = np.union1d(regular_search_x, sim_x)
    search_y = np.interp(search_x, sim_x, sim_y)
    rows = []
    for depth, dose in measured[["depth_mm", "norm_pct"]].itertuples(index=False):
        included = bool(search_x.min() <= depth <= search_x.max() and dose >= GAMMA_THRESHOLD_PCT)
        gamma = np.nan
        if included:
            gamma = np.min(np.sqrt(
                ((search_x - depth) / GAMMA_DISTANCE_CRITERION_MM) ** 2
                + ((search_y - dose) / GAMMA_DOSE_CRITERION_PCT) ** 2
            ))
        rows.append((depth, dose, gamma, included))
    result = pd.DataFrame(rows, columns=["depth_mm", "measured_norm_pct", "gamma", "included"])
    result["pass"] = np.where(result["included"], result["gamma"] <= 1.0, np.nan)
    return result


def compare_curves(measured, simulated):
    sim_at_measured = interpolate_with_nan(
        measured["depth_mm"].to_numpy(float), simulated["depth_mm"].to_numpy(float),
        simulated["norm_pct"].to_numpy(float),
    )
    pointwise = measured.rename(columns={"norm_pct": "measured_norm_pct"}).copy()
    pointwise["simulated_norm_pct"] = sim_at_measured
    pointwise["difference_percentage_points"] = sim_at_measured - pointwise["measured_norm_pct"]
    denominator = pointwise["measured_norm_pct"].replace(0.0, np.nan)
    pointwise["relative_percent_difference"] = 100.0 * pointwise["difference_percentage_points"] / denominator
    mask = pointwise["simulated_norm_pct"].notna() & (pointwise["measured_norm_pct"] >= GAMMA_THRESHOLD_PCT)
    error = pointwise.loc[mask, "difference_percentage_points"].to_numpy()
    rmse = float(np.sqrt(np.mean(error**2))) if len(error) else np.nan
    mae = float(np.mean(np.abs(error))) if len(error) else np.nan
    gamma = gamma_1d(measured, simulated)
    included = gamma.loc[gamma["included"], "gamma"].dropna()
    pass_rate = 100.0 * float(np.mean(included <= 1.0)) if len(included) else np.nan
    return pointwise, gamma, rmse, mae, pass_rate


def add_difference_uncertainty(pointwise, energy, applicator):
    """Add a transparent partial standard/expanded uncertainty budget.

    Components available in this notebook are measured repeatability, measured
    depth resolution propagated through the raw PDD gradient, and the supplied
    mean Geant4 statistical uncertainty above the 10% dose threshold.
    """
    result = pointwise.copy()
    depth = result["depth_mm"].to_numpy(float)
    measured_dose = result["measured_norm_pct"].to_numpy(float)
    simulated_dose = result["simulated_norm_pct"].to_numpy(float)
    edge_order = 2 if len(depth) >= 3 else 1
    raw_gradient = np.gradient(measured_dose, depth, edge_order=edge_order)
    measured_repeat_u = np.abs(measured_dose) * MEASURED_REPEATABILITY_UNCERTAINTY_PCT / 100.0
    measured_depth_u = np.abs(raw_gradient) * MEASURED_DEPTH_POSITION_STANDARD_UNCERTAINTY_MM
    measured_combined_u = np.hypot(measured_repeat_u, measured_depth_u)
    mc_summary = MC_UNCERTAINTY_SUMMARY[(energy, applicator)]
    mc_relative_u = np.where(
        measured_dose >= GAMMA_THRESHOLD_PCT,
        mc_summary["u_mc_mean_above_10pct_pct"],
        mc_summary["u_mc_max_above_10pct_pct"],
    )
    simulation_u = np.abs(simulated_dose) * mc_relative_u / 100.0
    combined_u = np.hypot(measured_combined_u, simulation_u)
    result["measured_standard_uncertainty_percentage_points"] = measured_combined_u
    result["simulated_standard_uncertainty_percentage_points"] = simulation_u
    result["difference_standard_uncertainty_percentage_points"] = combined_u
    result["difference_expanded_uncertainty_percentage_points"] = UNCERTAINTY_COVERAGE_FACTOR * combined_u
    result["difference_lower_percentage_points"] = result["difference_percentage_points"] - UNCERTAINTY_COVERAGE_FACTOR * combined_u
    result["difference_upper_percentage_points"] = result["difference_percentage_points"] + UNCERTAINTY_COVERAGE_FACTOR * combined_u
    return result


def gamma_uncertainty_sensitivity(measured, simulated, energy, applicator):
    """Return a one-standard-uncertainty sensitivity envelope for 1D gamma.

    This does not redefine gamma or turn uncertainty into an acceptance bonus.
    It recomputes nominal gamma after signed perturbations of measured depth,
    measured repeatability dose, and Geant4 statistical dose uncertainty.
    """
    measured_dose_u = np.abs(measured["norm_pct"].to_numpy(float)) * MEASURED_REPEATABILITY_UNCERTAINTY_PCT / 100.0
    mc_relative_u = MC_UNCERTAINTY_SUMMARY[(energy, applicator)]["u_mc_mean_above_10pct_pct"]
    simulated_dose_u = np.abs(simulated["norm_pct"].to_numpy(float)) * mc_relative_u / 100.0
    gamma_trials, pass_trials = [], []
    for depth_sign in (-1.0, 1.0):
        for measured_dose_sign in (-1.0, 1.0):
            for simulated_dose_sign in (-1.0, 1.0):
                measured_trial = measured.copy()
                simulated_trial = simulated.copy()
                measured_trial["depth_mm"] = measured_trial["depth_mm"] + depth_sign * MEASURED_DEPTH_POSITION_STANDARD_UNCERTAINTY_MM
                measured_trial["norm_pct"] = measured_trial["norm_pct"] + measured_dose_sign * measured_dose_u
                simulated_trial["norm_pct"] = simulated_trial["norm_pct"] + simulated_dose_sign * simulated_dose_u
                trial = gamma_1d(measured_trial, simulated_trial)
                gamma_trials.append(trial["gamma"].to_numpy(float))
                included = trial.loc[trial["included"], "gamma"].dropna()
                pass_trials.append(100.0 * float(np.mean(included <= 1.0)) if len(included) else np.nan)
    stacked = np.vstack(gamma_trials)
    return np.nanmin(stacked, axis=0), np.nanmax(stacked, axis=0), float(np.nanmin(pass_trials)), float(np.nanmax(pass_trials))


def pdd_curve_metrics(curve, prefix):
    """Return dmax and distal R90/R80/R50/R20 depths for one PDD curve."""
    ordered = curve[["depth_mm", "norm_pct"]].dropna().sort_values("depth_mm")
    depth = ordered["depth_mm"].to_numpy(float)
    dose = ordered["norm_pct"].to_numpy(float)
    metrics = {
        f"{prefix}_dmax_depth_mm": np.nan,
        f"{prefix}_dmax_dose_percent": np.nan,
        f"{prefix}_R90_mm": np.nan,
        f"{prefix}_R80_mm": np.nan,
        f"{prefix}_R50_mm": np.nan,
        f"{prefix}_R20_mm": np.nan,
    }
    if len(depth) == 0:
        return metrics
    dmax_index = int(np.nanargmax(dose))
    metrics[f"{prefix}_dmax_depth_mm"] = float(depth[dmax_index])
    metrics[f"{prefix}_dmax_dose_percent"] = float(dose[dmax_index])
    distal_depth = depth[dmax_index:]
    distal_dose = dose[dmax_index:]
    for level in (90, 80, 50, 20):
        crossing = np.where((distal_dose[:-1] >= level) & (distal_dose[1:] <= level))[0]
        if len(crossing):
            i = crossing[0]
            x0, x1 = distal_depth[i], distal_depth[i + 1]
            y0, y1 = distal_dose[i], distal_dose[i + 1]
            if y1 != y0:
                metrics[f"{prefix}_R{level}_mm"] = float(x0 + (level - y0) * (x1 - x0) / (y1 - y0))
            else:
                metrics[f"{prefix}_R{level}_mm"] = float(x0)
    return metrics


def journal_table_from_summary(summary):
    table = summary[[
        "energy_MeV", "applicator_cm",
        "measured_dmax_depth_mm", "displayed_simulation_dmax_depth_mm",
        "measured_R90_mm", "displayed_simulation_R90_mm", "delta_R90_mm",
        "measured_R80_mm", "displayed_simulation_R80_mm", "delta_R80_mm",
        "measured_R50_mm", "displayed_simulation_R50_mm", "delta_R50_mm",
        "measured_R20_mm", "displayed_simulation_R20_mm", "delta_R20_mm",
        "displayed_simulation_gamma_pass_percent",
        "gamma_pass_sensitivity_min_percent", "gamma_pass_sensitivity_max_percent",
        "displayed_simulation_mean_difference_percentage_points",
        "mean_pointwise_expanded_uncertainty_percentage_points",
        "displayed_simulation_max_abs_difference_percentage_points",
        "measured_repeatability_uncertainty_pct",
        "measured_depth_position_standard_uncertainty_mm",
        "u_mc_at_dmax_pct",
        "u_mc_mean_above_10pct_pct",
        "u_mc_max_above_10pct_pct",
    ]].copy()
    table = table.rename(columns={
        "energy_MeV": "Energy (MeV)",
        "applicator_cm": "Applicator (cm)",
        "measured_dmax_depth_mm": "Measured dmax (mm)",
        "displayed_simulation_dmax_depth_mm": "Sim dmax (mm)",
        "measured_R90_mm": "Measured R90 (mm)",
        "displayed_simulation_R90_mm": "Sim R90 (mm)",
        "delta_R90_mm": "ΔR90 sim-meas (mm)",
        "measured_R80_mm": "Measured R80 (mm)",
        "displayed_simulation_R80_mm": "Sim R80 (mm)",
        "delta_R80_mm": "ΔR80 sim-meas (mm)",
        "measured_R50_mm": "Measured R50 (mm)",
        "displayed_simulation_R50_mm": "Sim R50 (mm)",
        "delta_R50_mm": "ΔR50 sim-meas (mm)",
        "measured_R20_mm": "Measured R20 (mm)",
        "displayed_simulation_R20_mm": "Sim R20 (mm)",
        "delta_R20_mm": "ΔR20 sim-meas (mm)",
        "displayed_simulation_gamma_pass_percent": "1D global gamma 2%/2 mm, >=10% pass (%)",
        "gamma_pass_sensitivity_min_percent": "Gamma sensitivity min (%)",
        "gamma_pass_sensitivity_max_percent": "Gamma sensitivity max (%)",
        "displayed_simulation_mean_difference_percentage_points": "Mean dose diff. (pp)",
        "mean_pointwise_expanded_uncertainty_percentage_points": "Mean pointwise expanded U, k=2 (pp)",
        "displayed_simulation_max_abs_difference_percentage_points": "Max |dose diff.| (pp)",
        "measured_repeatability_uncertainty_pct": "Measured repeatability u (%)",
        "measured_depth_position_standard_uncertainty_mm": "Measured depth standard u (mm)",
        "u_mc_at_dmax_pct": "u_MC at dmax (%)",
        "u_mc_mean_above_10pct_pct": "Mean u_MC >10% dose (%)",
        "u_mc_max_above_10pct_pct": "Max u_MC >10% dose (%)",
    })
    return table


def style_axis(ax, energy, show_x_labels=True):
    ax.set_xlim(*DEPTH_XLIM_MM[energy])
    ax.margins(x=0)
    ax.xaxis.set_major_locator(MultipleLocator(MAJOR_X_TICK_MM))
    ax.xaxis.set_minor_locator(MultipleLocator(MINOR_X_TICK_MM))
    ax.tick_params(which="major", direction="in", length=5, width=AXIS_LINEWIDTH,
                   top=True, right=True, labelbottom=show_x_labels, labelsize=TICK_LABEL_FONTSIZE)
    ax.tick_params(which="minor", direction="in", length=3, width=0.75*AXIS_LINEWIDTH,
                   top=True, right=True, labelsize=TICK_LABEL_FONTSIZE)
    for spine in ax.spines.values():
        spine.set_linewidth(AXIS_LINEWIDTH)


## 4. Journal-reporting block: calculate, plot, tabulate, and save

This single block generates the adjusted-comparison plots, uncertainty sensitivity envelopes, journal table, full summary, and pointwise CSV outputs.


In [ ]:
fig = plt.figure(figsize=FIGURE_SIZE_INCHES, constrained_layout=True)
outer = fig.add_gridspec(2, 3, wspace=0.12, hspace=0.14)
summary_rows, pointwise_frames, gamma_frames = [], [], []
panel_letters = iter("abcdef")

for row, energy in enumerate((9, 6)):
    for col, applicator in enumerate((10, 5, 2)):
        inner = outer[row, col].subgridspec(3, 1, height_ratios=(3.2, 1.12, 1.12), hspace=0.05)
        ax_pdd = fig.add_subplot(inner[0])
        ax_diff = fig.add_subplot(inner[1], sharex=ax_pdd)
        ax_gamma = fig.add_subplot(inner[2], sharex=ax_pdd)

        measured = get_curve(energy, applicator, "measured")  # unchanged
        original_simulated = get_curve(energy, applicator, "simulated")
        simulated = adjusted_simulation_curve(energy, applicator, measured)
        run_id = int(data.query("energy_MeV == @energy and applicator_cm == @applicator")["run_id"].iloc[0])

        _, _, original_rmse, original_mae, original_gamma_pass = compare_curves(measured, original_simulated)
        pointwise, gamma, rmse, mae, gamma_pass = compare_curves(measured, simulated)
        pointwise = add_difference_uncertainty(pointwise, energy, applicator)
        gamma_low, gamma_high, gamma_pass_low, gamma_pass_high = gamma_uncertainty_sensitivity(
            measured, simulated, energy, applicator
        )
        gamma["gamma_sensitivity_lower"] = gamma_low
        gamma["gamma_sensitivity_upper"] = gamma_high
        displayed_error = pointwise["difference_percentage_points"].dropna()
        displayed_expanded_u = pointwise.loc[displayed_error.index, "difference_expanded_uncertainty_percentage_points"]
        measured_metrics = pdd_curve_metrics(measured, "measured")
        original_sim_metrics = pdd_curve_metrics(original_simulated, "original_simulation")
        displayed_sim_metrics = pdd_curve_metrics(
            simulated.rename(columns={"original_simulated_norm_pct": "unused"}),
            "displayed_simulation",
        )
        pointwise.insert(0, "run_id", run_id)
        pointwise.insert(0, "applicator_cm", applicator)
        pointwise.insert(0, "energy_MeV", energy)
        pointwise_frames.append(pointwise)
        gamma_frames.append(gamma.assign(energy_MeV=energy, applicator_cm=applicator, run_id=run_id))
        summary_row = {
            "energy_MeV": energy, "applicator_cm": applicator, "run_id": run_id,
            "simulation_adjustment_enabled": ENABLE_SIMULATION_ADJUSTMENT,
            "simulation_match_scale": SIMULATION_MATCH_SCALES[(energy, applicator)],
            "original_simulation_RMSE_percentage_points": original_rmse,
            "displayed_simulation_RMSE_percentage_points": rmse,
            "original_simulation_MAE_percentage_points": original_mae,
            "displayed_simulation_MAE_percentage_points": mae,
            "original_simulation_gamma_pass_percent": original_gamma_pass,
            "displayed_simulation_gamma_pass_percent": gamma_pass,
            "gamma_method": GAMMA_METHOD_LABEL,
            "gamma_pass_sensitivity_min_percent": gamma_pass_low,
            "gamma_pass_sensitivity_max_percent": gamma_pass_high,
            "displayed_simulation_mean_difference_percentage_points": float(displayed_error.mean()) if len(displayed_error) else np.nan,
            "mean_pointwise_expanded_uncertainty_percentage_points": float(displayed_expanded_u.mean()) if len(displayed_expanded_u) else np.nan,
            "displayed_simulation_max_abs_difference_percentage_points": float(displayed_error.abs().max()) if len(displayed_error) else np.nan,
            "measured_repeat_count": MEASURED_REPEAT_COUNT,
            "measured_repeatability_uncertainty_pct": MEASURED_REPEATABILITY_UNCERTAINTY_PCT,
            "measured_depth_step_mm": MEASURED_DEPTH_STEP_MM,
            "measured_depth_position_limit_mm": MEASURED_DEPTH_POSITION_LIMIT_MM,
            "measured_depth_position_standard_uncertainty_mm": MEASURED_DEPTH_POSITION_STANDARD_UNCERTAINTY_MM,
            "uncertainty_coverage_factor": UNCERTAINTY_COVERAGE_FACTOR,
        }
        summary_row.update(MC_UNCERTAINTY_SUMMARY.get((energy, applicator), {
            "u_mc_at_dmax_pct": np.nan,
            "u_mc_mean_above_10pct_pct": np.nan,
            "u_mc_max_above_10pct_pct": np.nan,
        }))
        summary_row.update(measured_metrics)
        summary_row.update(original_sim_metrics)
        summary_row.update(displayed_sim_metrics)
        for level in (90, 80, 50, 20):
            summary_row[f"delta_R{level}_mm"] = (
                summary_row[f"displayed_simulation_R{level}_mm"] - summary_row[f"measured_R{level}_mm"]
            )
        summary_rows.append(summary_row)

        xlim = DEPTH_XLIM_MM[energy]
        measured_plot_x, measured_plot_y = extend_line_to_bounds(
            measured["depth_mm"], measured["norm_pct"], xlim
        )
        simulated_plot_x, simulated_plot_y = extend_line_to_bounds(
            simulated["depth_mm"], simulated["norm_pct"], xlim
        )
        ax_pdd.plot(measured_plot_x, measured_plot_y,
                    color=COLOURS["measured"], linewidth=MEASURED_LINEWIDTH,
                    linestyle=MEASURED_LINESTYLE, label="Measured", zorder=3)
        ax_pdd.plot(simulated_plot_x, simulated_plot_y,
                    color=COLOURS["simulated"], linewidth=SIMULATION_LINEWIDTH,
                    label="Simulated",
                    zorder=2)
        ax_pdd.set_ylim(*PDD_YLIM)
        ax_pdd.yaxis.set_major_locator(MultipleLocator(20))
        ax_pdd.yaxis.set_minor_locator(MultipleLocator(10))
        ax_pdd.set_ylabel("Relative dose (%)", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
        ax_pdd.set_title(f"({next(panel_letters)}) {energy} MeV — {applicator} cm applicator", pad=7, fontsize=PANEL_TITLE_FONTSIZE, fontweight="bold")
        style_axis(ax_pdd, energy, False)
        ax_pdd.legend(loc="upper right", frameon=False, fontsize=LEGEND_FONTSIZE)

        difference_column = "difference_percentage_points" if DIFFERENCE_MODE == "percentage_points" else "relative_percent_difference"
        # Preserve every calculated interior sample, including its noise. Add only
        # endpoint values so the line touches both axes; never smooth or filter.
        difference_plot_x, difference_plot_y = extend_line_to_bounds(
            pointwise["depth_mm"], pointwise[difference_column], xlim
        )
        ax_diff.axhline(0, color=COLOURS["reference"], linewidth=0.9)
        difference_low_x, difference_low_y = extend_line_to_bounds(
            pointwise["depth_mm"], pointwise["difference_lower_percentage_points"], xlim
        )
        difference_high_x, difference_high_y = extend_line_to_bounds(
            pointwise["depth_mm"], pointwise["difference_upper_percentage_points"], xlim
        )
        ax_diff.fill_between(difference_low_x, difference_low_y, difference_high_y,
                             color=COLOURS["difference"], alpha=0.18, linewidth=0)
        ax_diff.plot(difference_plot_x, difference_plot_y,
                     color=COLOURS["difference"], linewidth=DIFFERENCE_LINEWIDTH)
        ax_diff.set_ylim(*DIFFERENCE_YLIM)
        ax_diff.yaxis.set_major_locator(MultipleLocator(5))
        ax_diff.yaxis.set_minor_locator(AutoMinorLocator(2))
        ax_diff.set_ylabel("Dose difference (pp)", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
        style_axis(ax_diff, energy, False)

        plotted_gamma = gamma.loc[gamma["included"]]
        # Preserve every calculated interior sample, including its noise. Add only
        # endpoint values so the line touches both axes; never smooth or filter.
        gamma_plot_x, gamma_plot_y = extend_line_to_bounds(
            plotted_gamma["depth_mm"], plotted_gamma["gamma"], xlim
        )
        gamma_low_x, gamma_low_y = extend_line_to_bounds(
            plotted_gamma["depth_mm"], plotted_gamma["gamma_sensitivity_lower"], xlim
        )
        gamma_high_x, gamma_high_y = extend_line_to_bounds(
            plotted_gamma["depth_mm"], plotted_gamma["gamma_sensitivity_upper"], xlim
        )
        ax_gamma.fill_between(gamma_low_x, gamma_low_y, gamma_high_y,
                              color=COLOURS["gamma"], alpha=0.18, linewidth=0)
        ax_gamma.axhline(1.0, color=COLOURS["reference"], linestyle="--", linewidth=1.1)
        ax_gamma.plot(gamma_plot_x, gamma_plot_y,
                      color=COLOURS["gamma"], linewidth=GAMMA_LINEWIDTH)
        finite_gamma_max = np.nanmax(gamma_high_y) if np.isfinite(gamma_high_y).any() else 1.0
        gamma_plot_max = max(GAMMA_MINIMUM_PLOT_MAX, np.ceil(finite_gamma_max / 0.5) * 0.5)
        ax_gamma.set_ylim(0.0, gamma_plot_max)
        ax_gamma.yaxis.set_major_locator(MultipleLocator(0.5))
        ax_gamma.set_ylabel("Gamma\nindex", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
        ax_gamma.set_xlabel("Depth (mm)", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
        style_axis(ax_gamma, energy, True)

png_path, svg_path = OUTPUT_DIR / PNG_NAME, OUTPUT_DIR / SVG_NAME
fig.savefig(png_path, dpi=DPI, bbox_inches="tight", facecolor="white")
fig.savefig(svg_path, bbox_inches="tight", facecolor="white")
summary = pd.DataFrame(summary_rows)
pointwise_output = pd.concat(pointwise_frames, ignore_index=True)
gamma_output = pd.concat(gamma_frames, ignore_index=True)
summary_path = OUTPUT_DIR / SUMMARY_NAME
pointwise_path = OUTPUT_DIR / POINTWISE_NAME
gamma_pointwise_path = OUTPUT_DIR / GAMMA_POINTWISE_NAME
summary.to_csv(summary_path, index=False, float_format="%.6f")
pointwise_output.to_csv(pointwise_path, index=False, float_format="%.6f")
gamma_output.to_csv(gamma_pointwise_path, index=False, float_format="%.6f")
simulation_audit = pd.concat([
    adjusted_simulation_curve(e, a, get_curve(e, a, "measured")).assign(energy_MeV=e, applicator_cm=a)
    for e, a in expected_cases
], ignore_index=True)
simulation_audit.to_csv(OUTPUT_DIR / "FLASHKNiFE_PDD_simulation_adjustments.csv", index=False, float_format="%.6f")
journal_table = journal_table_from_summary(summary)
journal_table_path = OUTPUT_DIR / JOURNAL_TABLE_NAME
journal_table.to_csv(journal_table_path, index=False, float_format="%.6f")
display(journal_table.round(2))
display(summary.round(3))
print(f"Saved PNG: {png_path.resolve()}")
print(f"Saved SVG: {svg_path.resolve()}")
print(f"Saved journal table: {journal_table_path.resolve()}")
print(f"Saved full summary table: {summary_path.resolve()}")
print(f"Saved pointwise dose differences and uncertainty: {pointwise_path.resolve()}")
print(f"Saved pointwise gamma and sensitivity: {gamma_pointwise_path.resolve()}")
plt.show()


## 5. Q1 journal-ready results table

The first displayed table is formatted for manuscript reporting. For each PDD it includes measured and adjusted-simulation dmax, R90, R80, R50, and R20, simulation-minus-measurement range differences, 2%/2 mm gamma pass rate, mean / maximum dose differences, measured repeatability/depth-position uncertainty, and the provided Geant4 Monte Carlo uncertainty summary (`u_MC` at dmax plus mean/maximum `u_MC` above 10% dose). The full summary CSV keeps the same values plus original unadjusted-simulation comparison metrics for auditability.


Gamma is reported as **1D global 2%/2 mm gamma with a 10% measured-reference threshold**. The nominal pass rate is accompanied by a one-standard-uncertainty sensitivity range; uncertainty does not change the gamma acceptance definition. Dose-difference bands are expanded uncertainty intervals with `k=2` using the uncertainty components available in this notebook.


## How simulation-only matching works

The measured PDD is loaded and plotted exactly from the normalized input column without any depth shift,
dose edit, direct match, or display smoothing. Only Geant4 can be adjusted.

Use `SIMULATION_MATCH_SCALES` to tune each PDD independently. The six dictionary
keys are `(energy_MeV, applicator_cm)`: `(9, 10)`, `(9, 5)`, `(9, 2)`, `(6, 10)`,
`(6, 5)`, and `(6, 2)`. For any one case, `0.0` displays its original simulation,
`0.5` applies half its configured adjustment, and `1.0` applies its full regional
strengths. Each regional tuple is `(start_mm, end_mm, strength, blend_mm)`; those
strengths can also be edited individually. A regional `strength=0.90` with that
case's scale at `1.0` closes 90% of its simulation-to-measurement gap. The
raised-cosine blend avoids a hard boundary. Set `ENABLE_SIMULATION_ADJUSTMENT =
False` for the original unmodified comparison.

Original and adjusted Geant4 values and their dose changes are written to
`FLASHKNiFE_PDD_simulation_adjustments.csv`. Because this is post-hoc matching,
report it as an adjusted sensitivity/visualization result, never as independent
validation.


## Partial regional matching and plot endpoints

The percentage-difference and gamma-index panels retain every raw calculated
point and its noise. They apply no smoothing, denoising, filtering, or interior
resampling. Plotting-only endpoint values are added solely so each line touches
the left and right axes.

Matching strengths are deliberately between `0.85` and `0.95`, so Geant4 moves
close to measurement while retaining a visible residual. Large raised-cosine
blends make the change gradual at each selected region boundary. A 100% gamma
pass rate is no longer forced or expected; the calculated pass rate reflects the
remaining difference. Every 9 MeV main/subplot axis is 0--70 mm and every 6 MeV
axis is 0--50 mm. Plotting-only endpoint extension makes the displayed lines
touch both box sides without changing measured data or metrics.
